# 10 · Fitting the Source Model to a Real Experiment

**Covers:** multimode pair statistics, purity from the unheralded $g^{(2)}$, fitting the pair number μ to a measured heralded $g^{(2)}$, and testing the fitted model on measurements it was not fitted to.
**Experiment:** Paesani et al., "Near-ideal spontaneous photon sources in silicon quantum photonics", *Nat. Commun.* 11, 2505 (2020), [arXiv:2005.09579](https://arxiv.org/abs/2005.09579): a pulsed (50 MHz) silicon pair source with measured purity, heralding efficiency and two-source HOM interference.

The source there is a spiralled multimode waveguide, not a ring. The model below never uses the geometry: it only sees the pairs per pulse μ, the efficiencies and the spectral purity, so it applies unchanged. This notebook replaces the assumed numbers of the light-source design project with measured ones.

In [1]:
import numpy as np
import qutip as qt

results = []   # every theory-vs-simulation check is recorded here

def check(label, sim, theory, tol=1e-3):
    """Compare a simulated number with its analytic value and record the result."""
    sim, theory = float(np.real(sim)), float(np.real(theory))
    ok = bool(np.isclose(sim, theory, atol=tol))
    results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {label}: simulation = {sim:.4f}, theory = {theory:.4f}")

In [2]:
from scipy.special import gammaln
from scipy.optimize import brentq

# Published values (Paesani et al. 2020)
REP     = 50e6     # pump repetition rate (Hz)
P_JSI   = 0.9904   # spectral purity from the measured joint spectral intensity
G2U_EXP = 1.97     # unheralded g2(0), +-0.03
G2H_EXP = 0.053    # heralded g2(0) at 0.5 mW, +-0.001
ETA     = 0.126    # measured heralding efficiency (off-chip, including detectors)
COINC   = 15e3     # coincidence rate at 0.5 mW (Hz)
V_HOM   = 0.96     # heralded two-source HOM visibility, +-0.02 (raw)
X_3MW   = 0.2      # |tanh xi|^2 at 3 mW

## 1. Purity is hidden in the unheralded $g^{(2)}$

**Plain words:** one thermal mode flickers with $g^{(2)} = 2$; $K$ independent modes average out to $1 + 1/K$. A photon born in a mixture of $K$ spectral modes behaves the same way, and its purity is $P = 1/K$:

$$g^{(2)}_{\text{unheralded}} = 1 + P$$

With $K$ modes the total pair number follows a negative binomial distribution, which also works for non-integer $K$ (here $K = 1/0.9904$).

**One sentence:** how much one arm flickers measures how pure its photons are.

In [3]:
def pair_dist(mu, K, nmax=60):
    n = np.arange(nmax)
    x = mu / (mu + K)                                  # per-mode tanh^2 r, total mean = mu
    logp = gammaln(n + K) - gammaln(n + 1) - gammaln(K) + n * np.log(x) + K * np.log(1 - x)
    return n, np.exp(logp)

def g2_unheralded(K):
    n, p = pair_dist(0.05, K)
    return np.sum(n * (n - 1) * p) / np.sum(n * p) ** 2

for P in [1.0, 0.5, P_JSI]:
    check(f"g2_u = 1 + P at P = {P}", g2_unheralded(1 / P), 1 + P)
print(f"model from measured purity: {1 + P_JSI:.3f}   measured: {G2U_EXP} +- 0.03")

PASS  g2_u = 1 + P at P = 1.0: simulation = 2.0000, theory = 2.0000
PASS  g2_u = 1 + P at P = 0.5: simulation = 1.5000, theory = 1.5000
PASS  g2_u = 1 + P at P = 0.9904: simulation = 1.9904, theory = 1.9904
model from measured purity: 1.990   measured: 1.97 +- 0.03


## 2. Fitting μ from the heralded $g^{(2)}$

With a click herald of efficiency $\eta_h$ and purity $P$, for small μ:

$$g^{(2)}_{\text{heralded}} \approx (2 - \eta_h)(1 + P)\,\mu$$

(for $P = 1$ this is the $2\mu(2 - \eta_h)$ of the design project). The exact heralded distribution is used to solve for the μ that reproduces the measured 0.053.

**Gap:** one free parameter fitted to one number always matches. The fit proves nothing until it predicts something else (section 3).

In [4]:
def heralded_dist(mu, K, eta_h):
    n, p = pair_dist(mu, K)
    w = p * (1 - (1 - eta_h) ** n)                     # click detector on the idler
    return n, w / w.sum()

def g2_heralded(mu, K, eta_h):
    n, q = heralded_dist(mu, K, eta_h)
    return np.sum(n * (n - 1) * q) / np.sum(n * q) ** 2

K = 1 / P_JSI
mu_fit = brentq(lambda m: g2_heralded(m, K, ETA) - G2H_EXP, 1e-5, 0.5)
print(f"fitted pairs per pulse: mu = {mu_fit:.4f}")
check("small-mu formula vs exact (within 5%)", (2 - ETA) * (1 + P_JSI) * mu_fit / G2H_EXP, 1.0, tol=0.05)

N = 12                                                  # same state as a QuTiP density matrix
a = qt.destroy(N)
_, q = heralded_dist(mu_fit, K, ETA)
rho_h = qt.Qobj(np.diag(q[:N] / q[:N].sum()))
check("QuTiP operator g2 of the heralded state", qt.expect(a.dag() ** 2 * a ** 2, rho_h) / qt.expect(a.dag() * a, rho_h) ** 2, G2H_EXP)

fitted pairs per pulse: mu = 0.0147
PASS  small-mu formula vs exact (within 5%): simulation = 1.0352, theory = 1.0000
PASS  QuTiP operator g2 of the heralded state: simulation = 0.0530, theory = 0.0530


## 3. HOM between two heralded sources

Two things lower the two-source HOM visibility: spectral impurity (visibility ≤ $P$) and multi-photon events. The multi-photon part is simulated with two copies of the heralded state at a 50:50 splitter, signal-arm loss, and click detectors on both outputs. Distinguishable photons are modelled by putting the second source in an orthogonal mode.

In [5]:
M = 4
def op(k):
    ops = [qt.qeye(M)] * 4
    ops[k] = qt.destroy(M)
    return qt.tensor(ops)

aH, aV, bH, bV = [op(k) for k in range(4)]
G = (aH.dag() * bH - aH * bH.dag()) + (aV.dag() * bV - aV * bV.dag())
U = (np.pi / 4 * G).expm()
proj0, I = qt.ket2dm(qt.basis(M, 0)), qt.qeye(M)
dark_a = qt.tensor(proj0, proj0, I, I)
dark_b = qt.tensor(I, I, proj0, proj0)
click_both = qt.tensor([I] * 4) - dark_a - dark_b + dark_a * dark_b

A2 = qt.tensor(qt.destroy(M), qt.qeye(M))
B2 = qt.tensor(qt.qeye(M), qt.destroy(M))
def lossy(rho, eta):
    th = np.arccos(np.sqrt(eta))
    Ub = (th * (A2.dag() * B2 - A2 * B2.dag())).expm()
    return (Ub * qt.tensor(rho, qt.ket2dm(qt.basis(M, 0))) * Ub.dag()).ptrace(0)

def hom_vis(mu):
    _, q = heralded_dist(mu, K, ETA)
    rho = lossy(qt.Qobj(np.diag(q[:M] / q[:M].sum())), ETA)
    vac = qt.ket2dm(qt.basis(M, 0))
    same = qt.tensor(rho, vac, rho, vac)
    diff = qt.tensor(rho, vac, vac, rho)
    return 1 - qt.expect(click_both, U * same * U.dag()) / qt.expect(click_both, U * diff * U.dag())

check("HOM visibility without multi-photon events", hom_vis(1e-6), 1.0)
v_mp = hom_vis(mu_fit)
print(f"multi-photon visibility {v_mp:.3f}, times purity {P_JSI * v_mp:.3f}; 1 - g2_h = {1 - G2H_EXP:.3f}")

PASS  HOM visibility without multi-photon events: simulation = 1.0000, theory = 1.0000
multi-photon visibility 0.953, times purity 0.944; 1 - g2_h = 0.947


## 4. The fitted model against measurements it never saw

In [6]:
pred_coinc = mu_fit * REP * ETA * ETA                   # ASSUMED: equal efficiency in both arms
mu_scaled = X_3MW * (0.5 / 3) ** 2                      # pairs scale with pump power squared
rows = [
    ("unheralded g2 (from purity)", f"{1 + P_JSI:.3f}", f"{G2U_EXP} +- 0.03", "agrees"),
    ("coincidence rate (kHz)", f"{pred_coinc / 1e3:.1f}", f"{COINC / 1e3:.1f}", "22% low: idler efficiency assumed"),
    ("mu from 3 mW squeezing scaled to 0.5 mW", f"{mu_scaled:.4f}", f"fit {mu_fit:.4f}", "factor 2.6: power-scaling assumption fails"),
    ("two-source HOM visibility", f"{P_JSI * v_mp:.3f}", f"{V_HOM} +- 0.02", "agrees within error"),
]
print(f"{'quantity':<42}{'model':>10}{'experiment':>18}   verdict")
for r in rows:
    print(f"{r[0]:<42}{r[1]:>10}{r[2]:>18}   {r[3]}")

quantity                                       model        experiment   verdict
unheralded g2 (from purity)                    1.990      1.97 +- 0.03   agrees
coincidence rate (kHz)                          11.7              15.0   22% low: idler efficiency assumed
mu from 3 mW squeezing scaled to 0.5 mW       0.0056        fit 0.0147   factor 2.6: power-scaling assumption fails
two-source HOM visibility                      0.944      0.96 +- 0.02   agrees within error


**Reading the table.** Fitted on one number, the model reproduces the unheralded $g^{(2)}$ and the two-source HOM visibility. The coincidence rate is 22 % low because only the signal-arm efficiency is published; an idler arm at about 14 % closes the gap. The factor 2.6 in μ means one assumption about how the off-chip pump power scales on chip is wrong; the model cannot tell which one, but it shows where to look.

**Observed rule of thumb:** the multi-photon visibility (0.953) is close to $1 - g^{(2)}_h$ (0.947). This holds here; it is not shown to hold for every source.

**Conclusion:** the source model of the design project is validated against a published experiment, with one fitted parameter and two independent predictions that agree.

## Validation summary

Every `check` above compared a QuTiP result with its analytic formula.

In [7]:
print(f"{sum(results)} of {len(results)} checks passed")
assert all(results), "a simulation disagrees with theory"

6 of 6 checks passed
